In [1]:
# =========================
# Cell 1) Imports
# =========================
from __future__ import annotations

import os
import re
import math
from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import List, Dict, Optional, Any, Tuple

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

In [2]:
# =========================
# Cell 2) Parameters & DB
# =========================
PROD_DAY = "20260128"     # YYYYMMDD
SHIFT_TYPE = "day"        # "day" | "night"

# ✅ 12시간 = 43200초, half-open [start, end)
WINDOW_SECONDS = 43200

STATIONS = ["Vision1", "Vision2"]
REMARKS  = ["PD", "Non-PD"]

# Ideal CT 기준일(고정)
IDEAL_BASE_PROD_DAY = "20260128"

# DB 환경변수로 override 가능
DB_CONFIG = {
    "host": os.getenv("PGHOST", "100.105.75.47"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
}

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine()

In [3]:
# =========================
# Cell 3) Utils: time parsing + window + interval (half-open)
# =========================

def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))

def _only_digits(s: str) -> str:
    return re.sub(r"[^0-9]", "", s)

def parse_hms_flexible(v: Any) -> time:
    """
    ✅ 유연 파서 (NameError 원인 해결: 여기서 정의)
    허용:
      - "HH:MI:SS"
      - "HH:MI"
      - "HH:MI:SS.xx" (소수초 제거)
      - "HHMMSS" / "HMMSS" / "HHMM"
      - time 객체
      - 숫자형(104000 등)
    """
    if v is None:
        raise ValueError("time value is None")

    if isinstance(v, time):
        return v

    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError(f"empty time value: {v}")

    # 소수초 제거
    if "." in s:
        s = s.split(".", 1)[0].strip()

    # 콜론 포맷
    if ":" in s:
        parts = s.split(":")
        if len(parts) == 2:
            hh, mm = parts
            ss = "0"
        else:
            hh, mm, ss = parts[0], parts[1], parts[2]
        return time(int(hh), int(mm), int(ss))

    # 숫자 포맷
    digits = _only_digits(s)
    if digits == "":
        raise ValueError(f"invalid time format: {v}")

    if len(digits) == 5:   # HMMSS
        digits = "0" + digits

    if len(digits) == 6:   # HHMMSS
        hh = int(digits[0:2])
        mm = int(digits[2:4])
        ss = int(digits[4:6])
        return time(hh, mm, ss)

    if len(digits) == 4:   # HHMM
        hh = int(digits[0:2])
        mm = int(digits[2:4])
        return time(hh, mm, 0)

    raise ValueError(f"unsupported time format: {v} (digits={digits})")

def build_shift_window(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    """
    ✅ half-open [start, end)
    day   = [08:30:00, 20:30:00)
    night = [20:30:00, 다음날 08:30:00)
    """
    d = yyyymmdd_to_date(prod_day)
    if shift_type == "day":
        start = datetime.combine(d, time(8, 30, 0), tzinfo=KST)
        end   = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
    elif shift_type == "night":
        start = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
        end   = datetime.combine(d + timedelta(days=1), time(8, 30, 0), tzinfo=KST)
    else:
        raise ValueError("SHIFT_TYPE must be 'day' or 'night'")

    dur = int((end - start).total_seconds())
    if dur != WINDOW_SECONDS:
        raise ValueError(f"Window seconds mismatch: got {dur}, expected {WINDOW_SECONDS}")
    return start, end

WINDOW_START, WINDOW_END = build_shift_window(PROD_DAY, SHIFT_TYPE)

@dataclass(frozen=True)
class Interval:
    start: datetime
    end: datetime   # ✅ end EXCLUSIVE

def merge_intervals(intervals: List[Interval]) -> List[Interval]:
    """half-open에서 겹치거나 맞닿는(cur.start <= last.end) 구간 합침"""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x.start)
    merged = [intervals[0]]
    for cur in intervals[1:]:
        last = merged[-1]
        if cur.start <= last.end:
            merged[-1] = Interval(last.start, max(last.end, cur.end))
        else:
            merged.append(cur)
    # 0길이 제거
    return [iv for iv in merged if iv.end > iv.start]

def overlap_seconds(a: Interval, b: Interval) -> int:
    """half-open overlap"""
    s = max(a.start, b.start)
    e = min(a.end, b.end)
    if e <= s:
        return 0
    return int((e - s).total_seconds())

def total_seconds(intervals: List[Interval]) -> int:
    return sum(int((iv.end - iv.start).total_seconds()) for iv in intervals)

WINDOW_START, WINDOW_END

(datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')),
 datetime.datetime(2026, 1, 28, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))

In [4]:
# =========================
# Cell 4) Table Names
# =========================
SAVE_SCHEMA = "i_daily_report"

T_REMARK_CHANGE_DAY   = "j_remark_change_day_daily"
T_REMARK_CHANGE_NIGHT = "j_remark_change_night_daily"

T_PLANNED_STOP_DAY    = "i_planned_stop_time_day_daily"
T_PLANNED_STOP_NIGHT  = "i_planned_stop_time_night_daily"

T_FINAL_AMT_DAY       = "a_station_day_daily_final_amount"
T_FINAL_AMT_NIGHT     = "a_station_night_daily_final_amount"

IDEAL_SCHEMA = "e1_FCT_ct"
IDEAL_TABLE  = "fct_whole_op_ct"

remark_change_table = T_REMARK_CHANGE_DAY if SHIFT_TYPE == "day" else T_REMARK_CHANGE_NIGHT
planned_stop_table  = T_PLANNED_STOP_DAY  if SHIFT_TYPE == "day" else T_PLANNED_STOP_NIGHT
final_amt_table     = T_FINAL_AMT_DAY     if SHIFT_TYPE == "day" else T_FINAL_AMT_NIGHT

remark_change_fqn = f'"{SAVE_SCHEMA}"."{remark_change_table}"'
planned_stop_fqn  = f'"{SAVE_SCHEMA}"."{planned_stop_table}"'
final_amt_fqn     = f'"{SAVE_SCHEMA}"."{final_amt_table}"'
ideal_fqn         = f'"{IDEAL_SCHEMA}"."{IDEAL_TABLE}"'

remark_change_fqn, planned_stop_fqn, final_amt_fqn, ideal_fqn

('"i_daily_report"."j_remark_change_day_daily"',
 '"i_daily_report"."i_planned_stop_time_day_daily"',
 '"i_daily_report"."a_station_day_daily_final_amount"',
 '"e1_FCT_ct"."fct_whole_op_ct"')

In [5]:
# =========================
# Cell 5) Column Introspection
# =========================
def get_columns(engine: Engine, schema: str, table: str) -> List[str]:
    sql = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema AND table_name = :table
        ORDER BY ordinal_position
    """)
    with engine.begin() as conn:
        rows = conn.execute(sql, {"schema": schema, "table": table}).fetchall()
    return [r[0] for r in rows]

cols_remark = get_columns(engine, SAVE_SCHEMA, remark_change_table)
cols_stop   = get_columns(engine, SAVE_SCHEMA, planned_stop_table)
cols_final  = get_columns(engine, SAVE_SCHEMA, final_amt_table)
cols_ideal  = get_columns(engine, IDEAL_SCHEMA, IDEAL_TABLE)

cols_remark, cols_stop, cols_final[:15], cols_ideal[:15]

(['prod_day',
  'station',
  'shift_type',
  'at_time',
  'from_remark',
  'to_remark',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'from_time',
  'to_time',
  'Total 계획 정지 시간',
  'total_planned_time',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'station',
  'pn',
  'remark',
  'A시간대(08:30:00 ~ 10:29:59)',
  'B시간대(10:30:00 ~ 12:29:59)',
  'C시간대(12:30:00 ~ 14:29:59)',
  'D시간대(14:30:00 ~ 16:29:59)',
  'E시간대(16:30:00 ~ 18:29:59)',
  'F시간대(18:30:00 ~ 20:29:59)',
  '합계',
  'updated_at'],
 ['id',
  'station',
  'remark',
  'month',
  'ct_eq',
  'uph',
  'final_ct',
  'created_at',
  'updated_at'])

In [6]:
# =========================
# Cell 6) Load Data (FIXED: station filter + planned_stop 포함)
# =========================
from sqlalchemy import text
import pandas as pd

def load_remark_changes() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {remark_change_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
        ORDER BY station, at_time
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "stations": list(STATIONS)   # ✅ ANY는 list/tuple 둘 다 OK
        })

def load_planned_stops() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {planned_stop_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE
        })

def load_final_amounts() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {final_amt_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "stations": list(STATIONS)
        })

df_remark = load_remark_changes()
df_stop   = load_planned_stops()
df_final  = load_final_amounts()

df_remark.head(), df_stop.head(), df_final.head()

(   prod_day  station shift_type   at_time from_remark to_remark  \
 0  20260128  Vision1        day  10:56:20      Non-PD        PD   
 1  20260128  Vision2        day  10:58:21      Non-PD        PD   
 
                         updated_at  
 0 2026-02-02 03:15:26.816824+00:00  
 1 2026-02-02 03:15:26.816824+00:00  ,
    prod_day shift_type from_time   to_time Total 계획 정지 시간  total_planned_time  \
 0  20260128        day                                10분                 600   
 1  20260128        day  10:50:00  11:00:00            10분                 600   
 
                  updated_at  
 0 2026-02-02 09:19:08+00:00  
 1 2026-02-02 09:19:08+00:00  ,
    prod_day shift_type  station        pn  remark A시간대(08:30:00 ~ 10:29:59)  \
 0  20260128        day  Vision1  35643009  Non-PD        PASS: 339, FAIL: 0   
 1  20260128        day  Vision1  35930927      PD                      None   
 2  20260128        day  Vision2  35643009  Non-PD        PASS: 374, FAIL: 0   
 3  20260128     

In [7]:
# =========================
# Cell 7) PASS Parse & Total Column Detect
# =========================
PASS_RE = re.compile(r"PASS\s*[:=]\s*(\d+)", re.IGNORECASE)

def parse_pass(x) -> int:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    m = PASS_RE.search(s)
    return int(m.group(1)) if m else 0

def detect_total_col(df: pd.DataFrame) -> str:
    if df.empty:
        return "합계"
    if "합계" in df.columns:
        return "합계"
    candidates = []
    for c in df.columns:
        if c.lower() in ("total", "sum", "total_amount", "overall", "grand_total"):
            candidates.append(c)
    for c in candidates:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    for c in df.columns:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    raise ValueError("Could not detect total column containing PASS/FAIL string.")

TOTAL_COL = detect_total_col(df_final)
TOTAL_COL

'합계'

In [8]:
# =========================
# Cell 8) Base remark (no change events) - FIXED
# =========================

DEFAULT_REMARK_FALLBACK = "Non-PD"  # 이벤트도 없고 remark도 못 찾으면 이걸로

def _clean_remark(v) -> Optional[str]:
    if v is None:
        return None
    s = str(v).strip()
    if s == "" or s.lower() == "nan" or s.lower() == "null":
        return None
    # 표준화
    if s.upper() == "PD":
        return "PD"
    if s.replace(" ", "").replace("_", "").replace("-", "").lower() in ("nonpd", "non-pd", "non_pd"):
        return "Non-PD"
    # 그 외 값이면 그대로 두되, 필요하면 여기서 걸러도 됨
    return s

def get_base_remark_for_station(df_final: pd.DataFrame, station: str) -> Optional[str]:
    s = df_final[df_final["station"] == station].copy()
    if s.empty or "remark" not in s.columns:
        return None

    cleaned = []
    for v in s["remark"].tolist():
        cv = _clean_remark(v)
        if cv is not None:
            cleaned.append(cv)

    vals = sorted(set(cleaned))
    if len(vals) == 0:
        return None
    if len(vals) > 1:
        # station에서 remark가 여러 개면, 실제로는 remark_change가 있어야 정상인데
        # 혹시 데이터가 섞였을 수 있음 -> 일단 첫 번째 사용 + 경고
        print(f"[WARN] station={station} has multiple remark values in final_amount: {vals} -> using first")
    return vals[0]

base_remark = {st: (get_base_remark_for_station(df_final, st) or DEFAULT_REMARK_FALLBACK) for st in STATIONS}
base_remark

[WARN] station=Vision1 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=Vision2 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first


{'Vision1': 'Non-PD', 'Vision2': 'Non-PD'}

In [9]:
# =========================
# Cell 9) Remark segments (FINAL: 43200 / half-open / boundary=at_time+1s)
# =========================

def _at_time_to_dt(at_time_val) -> datetime:
    """
    at_time(TEXT HH:MI:SS)를 KST datetime으로.
    night shift에서 자정 이후 시간은 날짜 +1.
    """
    at_t = parse_hms_flexible(at_time_val)  # ✅ Cell 3에 정의되어 NameError 해결
    at_dt = datetime.combine(WINDOW_START.date(), at_t, tzinfo=KST)
    if SHIFT_TYPE == "night" and at_dt < WINDOW_START:
        at_dt += timedelta(days=1)
    return at_dt

def build_remark_segments_for_station(df_remark: pd.DataFrame, station: str) -> List[tuple[str, Interval]]:
    """
    half-open [start, end)
    boundary = at_time + 1초
      이전: [cur_start, boundary) -> at_time 포함
      다음: [boundary, ...)
    """
    events = df_remark[df_remark["station"] == station].copy()
    events = events.sort_values("at_time")

    segs: List[tuple[str, Interval]] = []
    cur_start = WINDOW_START

    if events.empty:
        r0 = base_remark.get(station)
        if r0 is None:
            raise ValueError(f"No remark events and no base remark for station={station}")
        return [(r0, Interval(WINDOW_START, WINDOW_END))]

    cur_remark = str(events.iloc[0]["from_remark"])

    for _, row in events.iterrows():
        at_dt = _at_time_to_dt(row["at_time"])
        boundary = at_dt + timedelta(seconds=1)

        # clip to window
        if boundary < WINDOW_START:
            boundary = WINDOW_START
        if boundary > WINDOW_END:
            boundary = WINDOW_END

        if boundary > cur_start:
            segs.append((cur_remark, Interval(cur_start, boundary)))

        cur_start = boundary
        cur_remark = str(row["to_remark"])

        if cur_start >= WINDOW_END:
            break

    if cur_start < WINDOW_END:
        segs.append((cur_remark, Interval(cur_start, WINDOW_END)))

    # 0길이 제거 + 인접 동일 remark 병합
    segs = [(r, iv) for (r, iv) in segs if iv.end > iv.start]

    merged: List[tuple[str, Interval]] = []
    for r, iv in segs:
        if not merged:
            merged.append((r, iv))
            continue
        r_prev, iv_prev = merged[-1]
        if r_prev == r and iv_prev.end == iv.start:
            merged[-1] = (r_prev, Interval(iv_prev.start, iv.end))
        else:
            merged.append((r, iv))

    return merged

segments = {st: build_remark_segments_for_station(df_remark, st) for st in STATIONS}

# 검증: station별 합이 43200초
check_sum = {st: sum(int((iv.end - iv.start).total_seconds()) for _, iv in segments[st]) for st in STATIONS}
print("[CHECK] segment total seconds:", check_sum)

segments

[CHECK] segment total seconds: {'Vision1': 43200, 'Vision2': 43200}


{'Vision1': [('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 10, 56, 21, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 10, 56, 21, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul'))))],
 'Vision2': [('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 10, 58, 22, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 10, 58, 22, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul'))))]}

In [10]:
# =========================
# Cell 10) Planned stop union (skip missing/0 rows)
# =========================

def _is_missing_time(v) -> bool:
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return True
    if s in ("0", "0초", "00", "0000", "000000"):
        return True
    return False

def stops_to_intervals(df_stop: pd.DataFrame, station: Optional[str]) -> List[Interval]:
    d = df_stop.copy()
    if station is not None and "station" in d.columns:
        d = d[d["station"] == station]

    ivs: List[Interval] = []
    for _, row in d.iterrows():
        ft = row.get("from_time", None)
        tt = row.get("to_time", None)

        # ✅ from/to 비어있거나 0이면 실구간 없음 -> 스킵
        if _is_missing_time(ft) or _is_missing_time(tt):
            continue

        ft_t = parse_hms_flexible(ft)
        tt_t = parse_hms_flexible(tt)

        sdt = datetime.combine(WINDOW_START.date(), ft_t, tzinfo=KST)
        edt = datetime.combine(WINDOW_START.date(), tt_t, tzinfo=KST)

        if SHIFT_TYPE == "night":
            if sdt < WINDOW_START:
                sdt += timedelta(days=1)
            if edt < WINDOW_START:
                edt += timedelta(days=1)

        # 뒤집힘 방지
        if edt < sdt:
            sdt, edt = edt, sdt

        # half-open: 길이 0이면 제거
        if edt <= sdt:
            continue

        # window clip (원칙상 밖으로 안 나간다 했지만 방어)
        sdt = max(sdt, WINDOW_START)
        edt = min(edt, WINDOW_END)

        if edt > sdt:
            ivs.append(Interval(sdt, edt))

    return merge_intervals(ivs)

stop_union: Dict[str, List[Interval]] = {}
if "station" in df_stop.columns:
    for st in STATIONS:
        stop_union[st] = stops_to_intervals(df_stop, st)
else:
    common = stops_to_intervals(df_stop, None)
    for st in STATIONS:
        stop_union[st] = common

# 확인: station별 planned stop union 총합(초)
{st: total_seconds(ivs) for st, ivs in stop_union.items()}

{'Vision1': 600, 'Vision2': 600}

In [11]:
# =========================
# Cell 11) planned_stop_sec by station & remark
# =========================

def get_total_planned_time_if_available(df_stop: pd.DataFrame) -> Optional[int]:
    """
    from/to가 없는 행이 많을 수 있으므로, total_planned_time이 있으면 보조로 사용 가능.
    다만 이 값이 "행별 duration"인지 "shift 총합"인지 DB에 따라 다르니,
    여기서는 '이벤트가 아예 없는 station(remark 분해 불가)일 때'만 사용.
    """
    if "total_planned_time" not in df_stop.columns:
        return None
    s = df_stop.dropna(subset=["total_planned_time"])
    if s.empty:
        return None
    try:
        # shift 총합이라고 가정(첫 행) — 필요 시 sum으로 바꿔도 됨
        return int(float(s.iloc[0]["total_planned_time"]))
    except Exception:
        return None

total_planned_time_shift = get_total_planned_time_if_available(df_stop)

planned_sec_by_station_remark: Dict[Tuple[str, str], int] = {}

for st in STATIONS:
    has_events = not df_remark[df_remark["station"] == st].empty

    # init
    for r in REMARKS:
        planned_sec_by_station_remark[(st, r)] = 0

    if not has_events:
        # remark 변화 없으면 base_remark 하나만 43200을 커버하므로 planned도 그 remark에만 넣음
        r0 = base_remark.get(st)
        if r0 is None:
            raise ValueError(f"station={st}: base remark missing")

        if total_planned_time_shift is not None:
            planned_sec_by_station_remark[(st, r0)] = int(total_planned_time_shift)
        else:
            planned_sec_by_station_remark[(st, r0)] = int(total_seconds(stop_union[st]))
        continue

    # events 있는 경우: planned stop union을 remark segments에 overlap 분배
    for r, seg_iv in segments[st]:
        sec = 0
        for ps in stop_union[st]:
            sec += overlap_seconds(ps, seg_iv)
        planned_sec_by_station_remark[(st, r)] += sec

# 확인
planned_sec_by_station_remark

{('Vision1', 'PD'): 219,
 ('Vision1', 'Non-PD'): 381,
 ('Vision2', 'PD'): 98,
 ('Vision2', 'Non-PD'): 502}

In [12]:
# =========================
# Cell 12) Actual PASS by station & remark
# =========================
if df_final.empty:
    raise ValueError("final amount table returned 0 rows. Check prod_day/shift_type.")

df_final_work = df_final[["station", "pn", "remark", TOTAL_COL]].copy()
df_final_work["pass_qty"] = df_final_work[TOTAL_COL].apply(parse_pass)

actual_by_station_remark = (
    df_final_work.groupby(["station", "remark"], as_index=False)["pass_qty"].sum()
    .rename(columns={"pass_qty": "actual_pass_qty"})
)

actual_by_station_remark

,station,remark,actual_pass_qty
0,Vision1,Non-PD,418
1,Vision1,PD,1167
2,Vision2,Non-PD,440
3,Vision2,PD,1179


In [13]:
# =========================
# Cell 13) Ideal CT load (base day fixed)
# =========================
SIDE_TO_STATION = {"left": "Vision1", "right": "Vision2"}

def pick_col(cols: List[str], candidates: List[str], required=True) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    if required:
        raise ValueError(f"Missing columns, tried={candidates}, have={cols}")
    return None

ideal_col_station = pick_col(cols_ideal, ["station"])
ideal_col_remark  = pick_col(cols_ideal, ["remark"])
ideal_col_ct      = pick_col(cols_ideal, ["ct_eq", "final_ct", "ct"])
ideal_col_prod    = pick_col(cols_ideal, ["prod_day", "end_day", "day", "work_day"], required=False)
ideal_col_shift   = pick_col(cols_ideal, ["shift_type", "shift"], required=False)

where = []
params = {}

if ideal_col_prod is not None:
    where.append(f"{ideal_col_prod} = :base_prod_day")
    params["base_prod_day"] = IDEAL_BASE_PROD_DAY
if ideal_col_shift is not None:
    where.append(f"{ideal_col_shift} = :shift_type")
    params["shift_type"] = SHIFT_TYPE

where_sql = ("WHERE " + " AND ".join(where)) if where else ""

sql_ideal = text(f"""
    SELECT {ideal_col_station} AS side,
           {ideal_col_remark}  AS remark,
           MIN({ideal_col_ct}) AS ideal_ct_sec
    FROM {ideal_fqn}
    {where_sql}
    GROUP BY {ideal_col_station}, {ideal_col_remark}
""")

with engine.begin() as conn:
    df_ideal = pd.read_sql(sql_ideal, conn, params=params)

df_ideal["station"] = df_ideal["side"].astype(str).str.lower().map(SIDE_TO_STATION)
df_ideal

,side,remark,ideal_ct_sec,station
0,whole,Non-PD,NaN,NaN
1,left,Non-PD,15.18,Vision1
2,whole,PD,NaN,NaN
3,right,PD,17.67,Vision2
4,left,PD,18.35,Vision1
5,right,Non-PD,15.44,Vision2


In [14]:
# =========================
# Cell 14) OEE (Actual / Ideal) final
# =========================

def remark_window_seconds_for(station: str, remark: str) -> int:
    sec = 0
    for r, iv in segments[station]:
        if r == remark:
            sec += int((iv.end - iv.start).total_seconds())
    return sec

# 검증: station별 PD+Non-PD 합이 43200
sum_by_station = {st: sum(remark_window_seconds_for(st, r) for r in REMARKS) for st in STATIONS}
print("[CHECK] remark_window_sec sum:", sum_by_station)

rows = []
for st in STATIONS:
    for r in REMARKS:
        remark_window_sec = remark_window_seconds_for(st, r)
        if remark_window_sec <= 0:
            continue

        planned_sec = int(planned_sec_by_station_remark.get((st, r), 0))
        if planned_sec > remark_window_sec:
            print(f"[WARN] planned_sec > remark_window_sec: {st},{r} {planned_sec}>{remark_window_sec} -> cap")
            planned_sec = remark_window_sec

        effective_sec = remark_window_sec - planned_sec

        ideal_row = df_ideal[(df_ideal["station"] == st) & (df_ideal["remark"] == r)]
        ideal_ct = float(ideal_row.iloc[0]["ideal_ct_sec"]) if not ideal_row.empty else None

        act_row = actual_by_station_remark[
            (actual_by_station_remark["station"] == st) &
            (actual_by_station_remark["remark"] == r)
        ]
        actual = int(act_row.iloc[0]["actual_pass_qty"]) if not act_row.empty else 0

        ideal_qty = (effective_sec / ideal_ct) if (ideal_ct and ideal_ct > 0) else None

        # ✅ 너가 말하는 OEE = Actual / Ideal
        oee = (actual / ideal_qty) if (ideal_qty is not None and ideal_qty > 0) else None

        rows.append({
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "station": st,
            "remark": r,
            "remark_window_sec": remark_window_sec,
            "planned_stop_sec": planned_sec,
            "effective_sec": effective_sec,
            "ideal_ct_sec": ideal_ct,
            "ideal_qty": ideal_qty,
            "actual_pass_qty": actual,
            "oee_actual_over_ideal": oee,
            "updated_at": datetime.now(tz=KST),
        })

df_oee = pd.DataFrame(rows).sort_values(["station", "remark"]).reset_index(drop=True)
df_oee

[CHECK] remark_window_sec sum: {'Vision1': 43200, 'Vision2': 43200}


,prod_day,shift_type,station,remark,remark_window_sec,planned_stop_sec,effective_sec,ideal_ct_sec,ideal_qty,actual_pass_qty,oee_actual_over_ideal,updated_at
0,20260128,day,Vision1,Non-PD,8781,381,8400,15.18,553.359684,418,0.755386,2026-02-02 19:40:44.740084+09:00
1,20260128,day,Vision1,PD,34419,219,34200,18.35,1863.760218,1167,0.626154,2026-02-02 19:40:44.738150+09:00
2,20260128,day,Vision2,Non-PD,8902,502,8400,15.44,544.041451,440,0.808762,2026-02-02 19:40:44.742950+09:00
3,20260128,day,Vision2,PD,34298,98,34200,17.67,1935.483871,1179,0.609150,2026-02-02 19:40:44.741976+09:00


In [23]:
# =========================
# (추가) 통합 OEE (Ideal-가중) -> 3개 DF (퍼센트 문자열, 소수점 2자리 반올림)
# =========================
import numpy as np
import pandas as pd

required_cols = ["actual_pass_qty", "ideal_qty", "station", "remark"]
missing = [c for c in required_cols if c not in df_oee.columns]
if missing:
    raise ValueError(f"df_oee missing columns: {missing}")

g = df_oee.copy()
g["actual_pass_qty"] = pd.to_numeric(g["actual_pass_qty"], errors="coerce").fillna(0)
g["ideal_qty"] = pd.to_numeric(g["ideal_qty"], errors="coerce")

prod_day = PROD_DAY if "PROD_DAY" in globals() else None
shift_type = SHIFT_TYPE if "SHIFT_TYPE" in globals() else None
if prod_day is None or shift_type is None:
    raise ValueError("PROD_DAY / SHIFT_TYPE 변수가 필요합니다.")

def _ratio(num, den):
    den = float(den)
    if np.isnan(den) or den <= 0:
        return np.nan
    return float(num) / den

def to_pct_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return f"{x*100:.2f}%"

# 1) 전체 OEE
total_actual = g["actual_pass_qty"].sum()
total_ideal  = g["ideal_qty"].sum(skipna=True)
df_oee_total = pd.DataFrame([{
    "prod_day": prod_day,
    "shift_type": shift_type,
    "oee_total": to_pct_str(_ratio(total_actual, total_ideal)),
}])

# 2) station별 OEE
st = g.groupby("station", dropna=False).agg(
    actual=("actual_pass_qty", "sum"),
    ideal=("ideal_qty", "sum"),
).reset_index()
st["oee_station"] = st.apply(lambda r: to_pct_str(_ratio(r["actual"], r["ideal"])), axis=1)
df_oee_by_station = st.assign(prod_day=prod_day, shift_type=shift_type)[
    ["prod_day","shift_type","station","oee_station"]
]

# 3) remark별 OEE
rk = g.groupby("remark", dropna=False).agg(
    actual=("actual_pass_qty", "sum"),
    ideal=("ideal_qty", "sum"),
).reset_index()
rk["oee_remark"] = rk.apply(lambda r: to_pct_str(_ratio(r["actual"], r["ideal"])), axis=1)
df_oee_by_remark = rk.assign(prod_day=prod_day, shift_type=shift_type)[
    ["prod_day","shift_type","remark","oee_remark"]
]

display(df_oee_total)
display(df_oee_by_station)
display(df_oee_by_remark)

,prod_day,shift_type,oee_total
0,20260128,day,65.43%


,prod_day,shift_type,station,oee_station
0,20260128,day,Vision1,65.57%
1,20260128,day,Vision2,65.29%


,prod_day,shift_type,remark,oee_remark
0,20260128,day,Non-PD,78.18%
1,20260128,day,PD,61.75%
